In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup, Comment
from io import StringIO
import os
from datetime import date

# Today's date for dynamic filenames
today_str = date.today().isoformat()

# Base output folder (relative path since you're in "Baseball-Reference/")
BASE_DIR = os.path.join("Data")

# List of tables to scrape
tables_to_scrape = [
    {
        "url": "https://www.baseball-reference.com/leagues/majors/2025-standard-pitching.shtml",
        "table_id": "teams_standard_pitching",
        "folder": "team_pitching",
        "filename_prefix": "team_pitching"
    },
    {
        "url": "https://www.baseball-reference.com/leagues/majors/2025-standard-pitching.shtml",
        "table_id": "players_standard_pitching",
        "folder": "player_pitching",
        "filename_prefix": "player_pitching"
    },
    {
        "url": "https://www.baseball-reference.com/leagues/majors/2025-standard-batting.shtml",
        "table_id": "teams_standard_batting",
        "folder": "team_batting",
        "filename_prefix": "team_batting"
    },
    {
        "url": "https://www.baseball-reference.com/leagues/majors/2025-standard-batting.shtml",
        "table_id": "players_standard_batting",
        "folder": "player_batting",
        "filename_prefix": "player_batting"
    },
    {
        "url": "https://www.baseball-reference.com/leagues/majors/2025.shtml",
        "table_id": "team_output",
        "folder": "team_war",
        "filename_prefix": "team_war"
    },
    {
        "url": "https://www.baseball-reference.com/leagues/MLB-standings.shtml",
        "table_id": "expanded_standings_overall",
        "folder": "standings",
        "filename_prefix": "standings"
    }
]

def scrape_table(url, table_id):
    """
    Scrapes a table from the given URL using the specified table ID.
    Handles tables embedded within HTML comments.
    """
    print(f"Fetching data from: {url}")
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Attempt to find the table directly
    table = soup.find('table', {'id': table_id})

    # If not found, search within HTML comments
    if table is None:
        comments = soup.find_all(string=lambda text: isinstance(text, Comment))
        for comment in comments:
            if table_id in comment:
                comment_soup = BeautifulSoup(comment, 'html.parser')
                table = comment_soup.find('table', {'id': table_id})
                if table:
                    break

    if table is None:
        print(f"❌ Table with ID '{table_id}' not found.")
        return None

    # Read the table into a DataFrame
    df = pd.read_html(StringIO(str(table)))[0]

    # Clean the DataFrame
    if 'Rk' in df.columns:
        df = df[df['Rk'] != 'Rk']
        df.reset_index(drop=True, inplace=True)

    return df

def save_dataframe(df, subfolder, filename_prefix):
    """
    Saves the DataFrame to a CSV file under Data/<subfolder>/
    """
    full_folder = os.path.join(BASE_DIR, subfolder)
    os.makedirs(full_folder, exist_ok=True)

    filename = f"{filename_prefix}_{today_str}.csv"
    filepath = os.path.join(full_folder, filename)

    df.to_csv(filepath, index=False)
    print(f"✅ Data saved to {filepath}")

# Main execution
for table_info in tables_to_scrape:
    df = scrape_table(table_info["url"], table_info["table_id"])
    if df is not None:
        save_dataframe(df, table_info["folder"], table_info["filename_prefix"])


Fetching data from: https://www.baseball-reference.com/leagues/majors/2025-standard-pitching.shtml
❌ Table with ID 'teams_standard_pitching' not found.
Fetching data from: https://www.baseball-reference.com/leagues/majors/2025-standard-pitching.shtml
❌ Table with ID 'players_standard_pitching' not found.
Fetching data from: https://www.baseball-reference.com/leagues/majors/2025-standard-batting.shtml
❌ Table with ID 'teams_standard_batting' not found.
Fetching data from: https://www.baseball-reference.com/leagues/majors/2025-standard-batting.shtml
❌ Table with ID 'players_standard_batting' not found.
Fetching data from: https://www.baseball-reference.com/leagues/majors/2025.shtml
❌ Table with ID 'team_output' not found.
Fetching data from: https://www.baseball-reference.com/leagues/MLB-standings.shtml
❌ Table with ID 'expanded_standings_overall' not found.
